# Discrete solvers

> CEM, MPPI and projected gradient descent over discrete multi-agent actions.

In [ ]:
#| default_exp planning.solver.categorical

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
# Adapted from stable-worldmodel's planning package (https://github.com/galilai-group/stable-worldmodel):
#
# MIT License
#
# Copyright (c) 2026 GalilAI-group
#
# Permission is hereby granted, free of charge, to any person obtaining a copy
# of this software and associated documentation files (the "Software"), to deal
# in the Software without restriction, including without limitation the rights
# to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
# copies of the Software, and to permit persons to whom the Software is
# furnished to do so, subject to the following conditions:
#
# The above copyright notice and this permission notice shall be included in all
# copies or substantial portions of the Software.
#
# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
# IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
# FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
# AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
# LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
# OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
# SOFTWARE.

from __future__ import annotations

import time
from typing import Any

import numpy as np
import torch

from stable_marl.protocols import Costable
from stable_marl.planning.solver.base import SolverBase, gather_agents

In [ ]:
#| export
class CategoricalCEMSolver(SolverBase):
    """
    Cross-entropy method over discrete actions: one categorical distribution per (step, agent,
    action-block step), sampled with Gumbel-max, refit from the `topk` best candidates (per
    agent in ``'per_agent'`` mode). Candidates are one-hot ``(B, S, H, A, action_block * K)``;
    returns ``actions`` ``(n_envs, H, A, action_block)``.

    `smoothing` is added to the refit probabilities, `alpha` is their momentum.
    `init_action` is accepted for API parity and ignored, as in stable-worldmodel.
    """
    discrete = True

    def __init__(self, cost: Costable, batch_size: int | None = 1, num_samples: int = 300, n_steps: int = 30,
                 topk: int = 30, smoothing: float = 0.0, alpha: float = 0.0, **kwargs):
        super().__init__(cost, batch_size=batch_size, num_samples=num_samples, **kwargs)
        self.n_steps, self.topk, self.smoothing, self.alpha = n_steps, topk, smoothing, alpha

    def init_probs(self, n_envs: int) -> torch.Tensor:
        "Uniform probabilities (n_envs, H, A, action_block, K)."
        K = self._num_actions
        return torch.full((n_envs, self.horizon, self._num_agents, self.action_block, K), 1.0 / K,
                          dtype=self.dtype, device=self.device)

    def _sample_indices(self, probs: torch.Tensor) -> torch.Tensor:
        "Gumbel-max samples (B, S, H, A, action_block) of probabilities (B, H, A, action_block, K)."
        log_probs = probs.clamp_min(1e-10).log().unsqueeze(1).expand(probs.shape[0], self.num_samples, *probs.shape[1:])
        u = torch.rand(log_probs.shape, generator=self.torch_gen, device=self.device, dtype=self.dtype).clamp_min(1e-10)
        return (log_probs - (-u.log()).log()).argmax(dim=-1)

    def _weights(self, costs: torch.Tensor) -> torch.Tensor:
        "Refit weights (B, S, A), summing to 1 over S: uniform over the top-k of each agent."
        _, top = torch.topk(costs, k=self.topk, dim=1, largest=False)
        return torch.zeros_like(costs).scatter_(1, top, 1.0 / self.topk)

    @torch.inference_mode()
    def solve(self, info_dict: dict, init_action: Any = None) -> dict:
        start_time = time.time()
        n_envs = len(next(v for v in info_dict.values() if isinstance(v, (torch.Tensor, np.ndarray, list))))
        probs, final_costs = self.init_probs(n_envs), []
        for start, end in self._batches(n_envs):
            B, batch_probs, infos = end - start, probs[start:end], self._expand(info_dict, start, end)
            for _ in range(self.n_steps):
                indices = self._sample_indices(batch_probs)
                indices[:, 0] = batch_probs.argmax(dim=-1)              # the current mode is always a candidate
                one_hot = torch.nn.functional.one_hot(indices, num_classes=self._num_actions).to(self.dtype)
                candidates = one_hot.flatten(-2)                        # (B, S, H, A, action_block * K)
                costs = self._costs(self.cost.get_cost(dict(infos), candidates), B, self.num_samples)
                weights = self._weights(costs)
                new_probs = torch.einsum('bsa,bshajk->bhajk', weights, one_hot)
                if self.smoothing > 0:
                    new_probs = (new_probs + self.smoothing) / (new_probs + self.smoothing).sum(-1, keepdim=True)
                batch_probs = self.alpha * batch_probs + (1 - self.alpha) * new_probs if self.alpha > 0 else new_probs
            final_costs += self._cost_list((weights * costs).sum(dim=1))
            probs[start:end] = batch_probs
        self._log(start_time)
        return {'actions': probs.argmax(dim=-1).cpu(), 'costs': final_costs, 'probs': [probs.cpu()]}


class CategoricalMPPISolver(CategoricalCEMSolver):
    "Like :class:`CategoricalCEMSolver`, every candidate weighted by ``softmax(-cost / temperature)``."
    def __init__(self, cost: Costable, temperature: float = 1.0, **kwargs):
        super().__init__(cost, **kwargs)
        self.temperature = temperature

    def _weights(self, costs: torch.Tensor) -> torch.Tensor:
        return torch.softmax(-(costs - costs.min(dim=1, keepdim=True).values) / self.temperature, dim=1)

In [ ]:
#| export
class PGDSolver(SolverBase):
    """
    Projected gradient descent over relaxed discrete actions: each (step, agent, action-block step)
    is a point of the probability simplex over the K actions, optimised by gradient descent on the
    cost (which must be differentiable in the candidates) and projected back on the simplex.
    `num_samples` restarts are optimised; the best one is kept (per agent in ``'per_agent'``
    mode, where each agent's actions only receive the gradient of its own cost). Returns
    ``actions`` ``(n_envs, H, A, action_block)``. `init_action` may be action indices
    ``(n_envs, t, A, action_block)`` or relaxed actions ``(n_envs, t, A, action_block * K)``.
    """
    discrete = True

    def __init__(self, cost: Costable, n_steps: int, batch_size: int | None = None, var_scale: float = 1.0,
                 num_samples: int = 1, action_noise: float = 0.0, lr: float = 1.0, **kwargs):
        super().__init__(cost, batch_size=batch_size, num_samples=num_samples, **kwargs)
        self.n_steps, self.var_scale, self.action_noise, self.lr = n_steps, var_scale, action_noise, lr

    def _relaxed(self, init_action: torch.Tensor | None, n_envs: int) -> torch.Tensor:
        K, ab, A = self._num_actions, self.action_block, self._num_agents
        if init_action is not None and not init_action.is_floating_point():
            init_action = torch.nn.functional.one_hot(init_action.long(), K).to(self.dtype).flatten(-2)
        actions = torch.zeros(n_envs, self.horizon, A, ab * K, dtype=self.dtype, device=self.device)
        if init_action is not None:
            t = min(init_action.shape[1], self.horizon)
            actions[:, :t] = init_action[:, :t].to(self.device, self.dtype)
        actions = actions.unsqueeze(1).repeat_interleave(self.num_samples, dim=1)
        actions[:, 1:] += torch.randn(actions[:, 1:].shape, generator=self.torch_gen, device=self.device,
                                      dtype=self.dtype) * self.var_scale
        return self.project(actions)

    def project(self, actions: torch.Tensor) -> torch.Tensor:
        "Euclidean projection of each action-block step of `actions` (..., action_block * K) on the simplex."
        K = self._num_actions
        s = actions.reshape(-1, K)
        mu, _ = torch.sort(s, descending=True, dim=-1)
        cumulative = mu.cumsum(dim=-1)
        ks = torch.arange(1, K + 1, device=s.device, dtype=s.dtype)
        cond = (mu > (cumulative - 1) / ks).to(torch.int64)
        rho = (cond.cumsum(dim=-1) * cond).max(dim=-1, keepdim=True).values.clamp(min=1)
        psi = (cumulative.gather(-1, rho - 1) - 1) / rho
        return (s - psi).clamp(min=0.0).reshape(actions.shape)

    def solve(self, info_dict: dict, init_action: torch.Tensor | None = None) -> dict:
        start_time = time.time()
        n_envs = len(next(v for v in info_dict.values() if isinstance(v, (torch.Tensor, np.ndarray, list))))
        params, actions, final_costs = self._relaxed(init_action, n_envs), [], []
        for start, end in self._batches(n_envs):
            B, infos = end - start, self._expand(info_dict, start, end)
            x = params[start:end].clone().requires_grad_(True)
            optim = torch.optim.SGD([x], lr=self.lr)
            for _ in range(self.n_steps):
                costs = self._costs(self.cost.get_cost(dict(infos), x), B, self.num_samples)
                _backward(costs, x, self.mode)
                optim.step()
                optim.zero_grad(set_to_none=True)
                with torch.no_grad():
                    if self.action_noise > 0:
                        x += torch.randn(x.shape, generator=self.torch_gen, device=self.device, dtype=self.dtype) * self.action_noise
                    x.copy_(self.project(x))
            with torch.no_grad():
                costs = self._costs(self.cost.get_cost(dict(infos), x), B, self.num_samples)
                best = costs.argmin(dim=1, keepdim=True)                           # (B, 1, A)
                plan = gather_agents(x.detach(), best)[:, 0]                       # (B, H, A, ab * K)
            actions.append(plan.reshape(*plan.shape[:-1], self.action_block, self._num_actions).argmax(-1).cpu())
            final_costs += self._cost_list(costs.gather(1, best)[:, 0])
        self._log(start_time)
        return {'actions': torch.cat(actions), 'costs': final_costs}


def _backward(costs: torch.Tensor, params: torch.Tensor, mode: str):
    """
    Gradients of the cost (B, S, A) w.r.t. the candidates (B, S, H, A, X): of the joint cost, or
    in 'per_agent' mode, each agent's actions from its own cost only.
    """
    if mode == 'joint':
        costs[..., 0].sum().backward()
        return
    A = costs.shape[-1]
    grads = torch.zeros_like(params)
    for a in range(A):
        g, = torch.autograd.grad(costs[..., a].sum(), params, retain_graph=a < A - 1)
        grads[..., a, :] = g[..., a, :]
    params.grad = grads

### Tests

In [ ]:
from types import SimpleNamespace
import gymnasium as gym
from stable_marl.planning.solver.base import Solver, PLANNING_MODES

class TargetCost:
    "Distance of candidates (B, S, H, A, ab * K) to a target plan (H, A, ab) of action indices."
    def __init__(self, target, K, per_agent=False):
        self.target = torch.nn.functional.one_hot(target, K).float().flatten(-2)   # (H, A, ab * K)
        self.per_agent = per_agent
    def get_cost(self, info_dict, candidates):
        cost = (candidates - self.target).pow(2).sum(dim=(2, 4))                 # (B, S, A), differentiable
        return cost if self.per_agent else cost.sum(-1)
    def criterion(self, info_dict, candidates):
        return self.get_cost(info_dict, candidates)

K, cfg = 4, SimpleNamespace(horizon=3, action_block=2)
target = torch.tensor([[[0, 3], [2, 2]], [[1, 0], [3, 1]], [[2, 1], [0, 0]]])     # (H=3, A=2, ab=2)
space = gym.spaces.MultiDiscrete([K, K])
for mode in PLANNING_MODES:
    cost = TargetCost(target, K, per_agent=mode == 'per_agent')
    for solver in (CategoricalCEMSolver(cost, batch_size=3, num_samples=200, n_steps=10, topk=20, mode=mode),
                   CategoricalMPPISolver(cost, batch_size=3, num_samples=200, n_steps=10, temperature=0.5, mode=mode),
                   PGDSolver(cost, n_steps=30, num_samples=4, lr=0.2, mode=mode)):
        solver.configure(action_space=space, n_envs=3, config=cfg)
        assert isinstance(solver, Solver) and solver.action_dim == 2 * K
        out = solver.solve({'x': torch.zeros(3, 1)})
        assert out['actions'].shape == (3, 3, 2, 2) and (out['actions'] == target).all(), (type(solver).__name__, mode)
        assert len(out['costs']) == 3

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()